In [1]:
import pandas as pd
import numpy as np
import cv2

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from deepface import DeepFace

In [2]:


cap = cv2.VideoCapture("face.mp4")

time = 0
fps = cap.get(cv2.CAP_PROP_FPS)

results = []

while True:
    ret, frame = cap.read()
    if not ret:
        break

    result = DeepFace.analyze(frame, actions=['emotion'], enforce_detection=False)
    emotion = result[0]['dominant_emotion']

    results.append([time, emotion])
    time += 1.0 / fps

df = pd.DataFrame(results, columns=["time", "emotion"])
df.to_csv("emotion.csv", index=False)

In [3]:
# =========================
# ① データ読み込み
# =========================
log_df = pd.read_csv("log.csv")
emo_df = pd.read_csv("emotion.csv")

# =========================
# ② 時間を揃える（merge）
# =========================
# 近い時間でマージ（asof）
log_df = log_df.sort_values("time")
emo_df = emo_df.sort_values("time")

df = pd.merge_asof(log_df, emo_df, on="time")

# =========================
# ③ 特徴量抽出（時間窓）
# =========================
WINDOW_SIZE = 1.0  # 秒

features = []
labels = []

start_time = df["time"].min()
end_time = df["time"].max()

t = start_time

while t < end_time:
    window = df[(df["time"] >= t) & (df["time"] < t + WINDOW_SIZE)]

    if len(window) < 5:
        t += WINDOW_SIZE
        continue

    # ===== 特徴量 =====
    jump_count = window["isJump"].sum()

    avg_dist = window["dist"].mean()
    min_dist = window["dist"].min()

    # 危険ジャンプ（距離が小さいときにジャンプ）
    risky_jump = window[(window["isJump"] == 1) & (window["dist"] < 0.2)]
    risky_jump_count = len(risky_jump)

    velocity_var = window["velocityY"].var()

    # ===== ラベル =====
    # 最頻出の感情を採用
    label = window["emotion"].mode()
    if len(label) == 0:
        t += WINDOW_SIZE
        continue

    label = label[0]

    features.append([
        jump_count,
        avg_dist,
        min_dist,
        risky_jump_count,
        velocity_var
    ])
    labels.append(label)

    t += WINDOW_SIZE

# =========================
# ④ 学習
# =========================
X = np.array(features)
y = np.array(labels)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestClassifier()
model.fit(X_train, y_train)

# =========================
# ⑤ 評価
# =========================
y_pred = model.predict(X_test)

print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

# =========================
# ⑥ 特徴量重要度
# =========================
feature_names = [
    "jump_count",
    "avg_dist",
    "min_dist",
    "risky_jump_count",
    "velocity_var"
]

importances = model.feature_importances_

print("\n=== Feature Importance ===")
for name, imp in zip(feature_names, importances):
    print(f"{name}: {imp:.3f}")

FileNotFoundError: [Errno 2] No such file or directory: 'log.csv'